# BTC Training Data Preparation
This notebook prepares the Bitcoin dataset for machine learning by calculating technical indicators using NumPy and Pandas.

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Load Data
data_path = os.path.join('data', 'btcusdt_analysis_data.csv')
df = pd.read_csv(data_path)

# Convert Timestamp to Datetime and set as index
df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Datetime', inplace=True)
df.sort_index(inplace=True)

print(f"Data loaded: {len(df)} rows.")

## Technical Indicators Implementation
We use NumPy and Pandas operations to calculate indicators that help classify trends.

In [ ]:
def calculate_indicators(df):
    """
    Calculates technical indicators for trend classification using NumPy/Pandas.
    """
    close = df['Close'].values
    high = df['High'].values
    low = df['Low'].values
    
    # --- 1. RSI (Relative Strength Index) ---
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # --- 2. Stochastic RSI ---
    rsi = df['RSI']
    stoch_rsi_min = rsi.rolling(window=14).min()
    stoch_rsi_max = rsi.rolling(window=14).max()
    df['StochRSI'] = (rsi - stoch_rsi_min) / (stoch_rsi_max - stoch_rsi_min)
    
    # --- 3. MACD (Moving Average Convergence Divergence) ---
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']
    
    # --- 4. ADX (Average Directional Index) ---
    # Needs True Range, +DM, -DM
    plus_dm = df['High'].diff()
    minus_dm = df['Low'].diff()
    plus_dm[plus_dm < 0] = 0
    minus_dm[minus_dm > 0] = 0
    
    tr1 = df['High'] - df['Low']
    tr2 = abs(df['High'] - df['Close'].shift(1))
    tr3 = abs(df['Low'] - df['Close'].shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    
    atr = tr.rolling(window=14).mean()
    plus_di = 100 * (plus_dm.rolling(window=14).mean() / atr)
    minus_di = 100 * (abs(minus_dm.rolling(window=14).mean()) / atr)
    dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    df['ADX'] = dx.rolling(window=14).mean()
    
    # --- 5. Trend Indicators (EMA) ---
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()
    
    # --- 6. Bollinger Bands ---
    sma20 = df['Close'].rolling(window=20).mean()
    std20 = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = sma20 + (std20 * 2)
    df['BB_Lower'] = sma20 - (std20 * 2)
    
    return df

# Calculate indicators
df = calculate_indicators(df)

# Drop rows with NaN values resulting from rolling calculations
df.dropna(inplace=True)

print(f"Indicators calculated. Remaining rows: {len(df)}")
print(df.tail())

## Data Ready for Feature Engineering
The dataframe now contains technical indicators and is ready for model input preparation.

In [ ]:
# Inspect final dataframe
print("Dataframe ready. Shape:", df.shape)
print(df.tail())